# SPARKS Expense Assistant: hosted Qwen + embeddings on a Colab GPU

This notebook turns a Colab GPU into the **model server** for the expense assistant on your laptop.
It serves two things behind **one HTTPS URL** protected by an API key:

| What | Model | Used by the app for |
|---|---|---|
| Chat + tool calling (`/v1/chat/completions`) | `Qwen/Qwen3-4B-Instruct-2507` via **vLLM** | Policy answers now; planner, executor and critic agents next |
| Embeddings (`/v1/embeddings`) | `BAAI/bge-small-en-v1.5` on the GPU | Vector half of hybrid policy search |

```
 Colab GPU (L4)                                                     Your laptop
 ┌────────────────────────────────────────────────┐
 │ vLLM            127.0.0.1:8000  Qwen3-4B        │
 │   ▲                                            │
 │ gateway (FastAPI) 127.0.0.1:8090               │      HTTPS + API key
 │   · checks the API key                         │◄──────────────────────── expense assistant API
 │   · /v1/embeddings  → bge-small on GPU         │   https://<tunnel URL>/v1
 │   · /v1/chat/completions → vLLM                │
 │   ▲                                            │
 │ tunnel: cloudflared (random URL) or ngrok      │
 └────────────────────────────────────────────────┘
```

## ▶ After any disconnect or restart: Runtime → Run all

**You never need to run steps one by one.** Every step checks what is already there and reuses it:

| Situation | What Run all does |
|---|---|
| Reconnected to the **same machine** (step 1 says "Same machine") | Skips the install and download, waits for a vLLM that is still loading, keeps the gateway and **keeps the same tunnel URL** if it still works. About 1 minute. |
| **Fresh machine** (deleted or timed-out runtime) | Asks once for Google Drive access. Reinstalls from the **package cache on Drive** (no 3–4 GB download; unpacking still takes a couple of minutes), reuses the **API key saved on Drive**, downloads the model from Hugging Face (1–2 min), then starts everything. Expect about 4–7 minutes. |
| A server dies while the runtime is alive | The monitor (step 8) restarts it by itself, and prints a new `.env` block if the URL changed. |

**Keep the same URL and key, so `.env` never changes:** add `COLAB_API_KEY` (any long random string) to Colab **Secrets**. For the URL, use ngrok's free static domain: set `TUNNEL = "ngrok"` in step 1, and add `NGROK_AUTHTOKEN` and `NGROK_DOMAIN` (for example `your-name.ngrok-free.app`, from the ngrok dashboard) as Secrets. Quick cloudflared tunnels get a new random URL on every fresh machine.

**First time**
1. **Runtime → Change runtime type → L4 GPU** (A100 also works).
2. Optional Secrets (key icon on the left): `COLAB_API_KEY`, `HF_TOKEN` (faster Hugging Face downloads), and `NGROK_AUTHTOKEN` + `NGROK_DOMAIN` for a fixed URL.
3. **Runtime → Run all.** The first run takes about 8–12 minutes.
4. Copy the `.env` block printed in step 7 into the project's `.env`. Then on your laptop run `.venv\Scripts\python.exe -m scripts.check_hosted_model` and restart the app (`run.bat`).

**Avoiding disconnects**
- Keep this tab open while the first run finishes. With Colab Pro+, turn on **background execution** where Colab offers it.
- *"Your session crashed after using all available RAM"*: choose **High-RAM** under **Runtime → Change runtime type**.
- Check **compute units** on Colab's resources panel. When they run out, Colab ends GPU runtimes.

**Good to know**
- When the runtime stops, the app keeps working: policy search falls back to local embeddings, but chat answers show an error until this notebook runs again.
- Use **synthetic data only**. The URL is protected by the API key, but don't send real employee or receipt data.
- The GPU uses Colab compute units. Use **Runtime → Disconnect and delete runtime** when you're done.

## 1. Settings and helpers

Holds everything Colab forgets after a restart: settings, helpers, GPU type and API key.
The generated API key and the tunnel URL are saved under `/content`, so re-running on the same machine reuses them.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"   # alternatives: "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"  # same as the app's local fallback; "BAAI/bge-base-en-v1.5" also fits
MAX_MODEL_LEN = 32768                      # context window; lower to 16384 if vLLM runs out of memory
GPU_MEMORY_UTILIZATION = 0.80              # vLLM's share of GPU memory; the rest is for the embedding model
TUNNEL = "cloudflared"                     # "cloudflared" (no account, random URL) or "ngrok" (fixed URL with
                                           #   NGROK_AUTHTOKEN + NGROK_DOMAIN secrets)
MAX_OUTPUT_TOKENS = 4096                   # written into the .env block for the app
FAST_START = True        # --enforce-eager: skips torch.compile + CUDA-graph capture; starts 1-3 min sooner,
                         # generation a little slower (fine for a few users)
# Google Drive checkpoints (folder MyDrive/colab_expense_assistant). A fresh runtime asks once for Drive access.
USE_DRIVE = True              # keep the API key and the black-box log on Drive (a few KB)
DRIVE_CACHE_PACKAGES = True   # pip download cache (~3-4 GB): a fresh runtime reinstalls vLLM without downloading it
DRIVE_CACHE_MODEL = False     # model copy (~8 GB): Colab downloads it from Hugging Face about as fast, so only
                              # turn this on if Hugging Face is slow AND Drive has the space (15 GB free Drive
                              # fits the packages OR the model, not both)
VLLM_PORT, GATEWAY_PORT = 8000, 8090      # never 8080: Colab's own runtime server listens there
GATEWAY = f"http://127.0.0.1:{GATEWAY_PORT}"
VLLM_HEALTH = f"http://127.0.0.1:{VLLM_PORT}/health"
MODEL_DIR = f"/content/models/{MODEL_ID.split('/')[-1]}"                 # local copy that vLLM loads
DRIVE_ROOT = "/content/drive/MyDrive/colab_expense_assistant"
DRIVE_DIR = f"{DRIVE_ROOT}/models/{MODEL_ID.split('/')[-1]}"
KEY_FILE, URL_FILE = "/content/.colab_api_key", "/content/.tunnel_url"
DRIVE_KEY_FILE = f"{DRIVE_ROOT}/api_key"

import json, os, re, secrets, shutil, subprocess, sys, time
from pathlib import Path
import requests


def is_up(url, headers=None):
    try:
        return requests.get(url, timeout=5, headers=headers).ok
    except requests.RequestException:
        return False


def require(*names, step):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise SystemExit(f"Missing {', '.join(missing)}: run step {step} first (Colab forgets variables after a restart).")


def importable(*modules):
    """True when the kernel's own Python can import the modules (the one %pip installs into)."""
    return subprocess.run([sys.executable, "-c", "import " + ", ".join(modules)], capture_output=True).returncode == 0


def pids(pattern):
    out = subprocess.run(["pgrep", "-f", pattern], capture_output=True, text=True).stdout.split()
    return [int(p) for p in out if int(p) != os.getpid()]


def age_seconds(pid):
    out = subprocess.run(["ps", "-o", "etimes=", "-p", str(pid)], capture_output=True, text=True).stdout.strip()
    return int(out) if out.isdigit() else 0


def stop_processes(pattern):
    """Stop OUR leftovers from an earlier run, matched by command line (for example a stuck vLLM that
    still holds GPU memory). Never kills by port: Colab's own runtime server shares this machine."""
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
    time.sleep(3)


def port_in_use(port):
    import socket
    with socket.socket() as sock:
        sock.settimeout(1)
        return sock.connect_ex(("127.0.0.1", port)) == 0


def require_free_port(port, setting):
    if port_in_use(port):
        raise SystemExit(f"Port {port} is used by another program (Colab itself uses 8080). Set {setting} to a "
                         "free port in step 1, then Run all.")


def tail(path, chars=4000):
    try:
        return open(path).read()[-chars:]
    except FileNotFoundError:
        return f"({path} not found)"


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:          # secret not set, or notebook access not granted
        return None


# GPU and dtype
cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if not cap:
    raise SystemExit("No GPU found. Use Runtime -> Change runtime type -> L4 GPU, then Run all again.")
DTYPE = "bfloat16" if float(cap.splitlines()[0]) >= 8 else "half"   # T4 (7.5) has no bfloat16
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip(), f"| compute {cap} -> dtype {DTYPE}")
ram = subprocess.run(["free", "-g"], capture_output=True, text=True).stdout.splitlines()[1].split()
print(f"RAM {ram[1]} GB total, {ram[-1]} GB available | disk free {shutil.disk_usage('/content').free / 1e9:.0f} GB")
print("Same machine as before:" if os.path.exists(KEY_FILE) else "Fresh machine:",
      "model on disk" if os.path.exists(os.path.join(MODEL_DIR, ".complete")) else "model not downloaded yet",
      "| vLLM running" if is_up(VLLM_HEALTH) else "| vLLM not running")

# Google Drive checkpoints
DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
if USE_DRIVE and not DRIVE_OK:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
    except Exception as exc:                   # access refused, or not running in Colab
        print("Google Drive not mounted (" + type(exc).__name__ + "); continuing without Drive checkpoints.")
DRIVE_FREE_GB = shutil.disk_usage("/content/drive/MyDrive").free / 1e9 if DRIVE_OK else 0.0
if DRIVE_OK:
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Google Drive: {DRIVE_FREE_GB:.1f} GB free | checkpoints in {DRIVE_ROOT}")
    if DRIVE_CACHE_PACKAGES:
        pip_cache = f"{DRIVE_ROOT}/pip-cache"
        if os.path.isdir(pip_cache) or DRIVE_FREE_GB > 5:
            os.environ["PIP_CACHE_DIR"] = pip_cache    # %pip in step 2 reads and fills this cache
            print("Package cache on Drive:", pip_cache)
        else:
            print(f"Package cache skipped: needs about 5 GB free on Drive, {DRIVE_FREE_GB:.1f} GB available.")

# API key: Colab secret, else the key saved on Drive, else the key from earlier on this machine, else a new one
API_KEY = colab_secret("COLAB_API_KEY")
if API_KEY:
    print("API key: COLAB_API_KEY from Colab Secrets (stays the same on every machine).")
elif DRIVE_OK and os.path.exists(DRIVE_KEY_FILE):
    API_KEY = Path(DRIVE_KEY_FILE).read_text().strip()
    print("API key: reusing the key saved on Google Drive (same key on every machine).")
elif os.path.exists(KEY_FILE):
    API_KEY = Path(KEY_FILE).read_text().strip()
    print("API key: reusing the key generated earlier on this machine.")
else:
    API_KEY = secrets.token_urlsafe(32)
    print("API key: generated a new one (printed in the .env block in step 7).")
with open(KEY_FILE, "w") as handle:            # also marks "same machine" for the next run
    handle.write(API_KEY)
os.chmod(KEY_FILE, 0o600)
if DRIVE_OK and not os.path.exists(DRIVE_KEY_FILE):
    Path(DRIVE_KEY_FILE).write_text(API_KEY)   # private to your Drive; reused by the next fresh runtime
AUTH = {"Authorization": f"Bearer {API_KEY}"}

if colab_secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = colab_secret("HF_TOKEN")
    print("Using HF_TOKEN from Colab Secrets.")

## 2. Install

`%pip` installs into the same Python this notebook runs on, which the later steps use to start the servers.
The embedding server's packages go in first, then vLLM, so vLLM's pinned `torch`/`transformers` versions win.
The first install takes about 3–5 minutes; it is **skipped** when everything is already installed.

Red **"pip's dependency resolver … incompatible"** lines are expected on Colab. They are about Colab's own preinstalled packages (google-colab, gradio, cudf, …), which this notebook doesn't use. You can ignore them, and you don't need to restart the session afterwards.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

PACKAGES = ("vllm", "sentence_transformers", "fastapi", "uvicorn", "httpx", "huggingface_hub")
if importable(*PACKAGES):
    print("Already installed; skipping.")
else:
    %pip install -q sentence-transformers fastapi uvicorn httpx
    %pip install -q vllm
if not importable(*PACKAGES):
    raise SystemExit("The packages are still not importable by " + sys.executable + "; see the pip output above.")
print(subprocess.run([sys.executable, "-c", "import vllm, torch, sentence_transformers as st; print('vllm', vllm.__version__, "
                      "'| torch', torch.__version__, '| sentence-transformers', st.__version__)"],
                     capture_output=True, text=True).stdout.strip())

## 3a. Download the model (with progress)

Downloads the ~8 GB model to local disk once, with a progress bar. An interrupted download **resumes**.
With `DRIVE_CACHE_MODEL = True` it also keeps an 8 GB copy in Google Drive, if Drive has room, and a fresh machine copies from there. That's off by default: Colab downloads from Hugging Face about as fast, and 15 GB of Drive fits the package cache or the model, not both.
Skipped when the model is already on disk.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

def ensure_model():
    done_marker = os.path.join(MODEL_DIR, ".complete")
    if os.path.exists(done_marker):
        print("Model already on local disk:", MODEL_DIR)
        return
    if DRIVE_OK and DRIVE_CACHE_MODEL and os.path.exists(os.path.join(DRIVE_DIR, ".complete")):
        print("Copying the model from Google Drive (faster than downloading) ...")
        started = time.time()
        shutil.copytree(DRIVE_DIR, MODEL_DIR, dirs_exist_ok=True)
        print(f"Copied in {time.time() - started:.0f}s")
        return
    from huggingface_hub import snapshot_download
    started = time.time()
    snapshot_download(MODEL_ID, local_dir=MODEL_DIR,
                      allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken", "*.py"])
    Path(done_marker).write_text(MODEL_ID)
    print(f"Downloaded in {time.time() - started:.0f}s")
    if DRIVE_OK and DRIVE_CACHE_MODEL:
        size_gb = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file()) / 1e9
        free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
        if free_gb < size_gb + 1:
            print(f"Not copying the model to Drive: needs {size_gb + 1:.1f} GB, Drive has {free_gb:.1f} GB free.")
        else:
            print(f"Saving a {size_gb:.1f} GB copy to Google Drive for the next fresh runtime ...")
            shutil.copytree(MODEL_DIR, DRIVE_DIR, dirs_exist_ok=True)


ensure_model()
size = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file())
print(f"Model ready: {size / 1e9:.1f} GB in {MODEL_DIR}")

## 3b. Start Qwen with vLLM

vLLM runs as a **background process**, so it keeps loading even if the browser disconnects.
If it is already running, this does nothing; if it is **still loading** from before a disconnect, this waits for it instead of restarting it.
Progress is printed every 30 seconds. On an L4 with `FAST_START = True`, expect about 1–3 minutes once the model is on disk.
vLLM uses its own sampler, so it skips the slow FlashInfer compile ("Building JIT module sampling"). A **black-box log** records RAM, GPU memory and the server logs every 10 s; it goes to Google Drive (`USE_DRIVE = True`), so it survives a deleted runtime.

vLLM listens only on `127.0.0.1`, so it is never exposed directly. The gateway in step 5 is the only way in.
`--enable-auto-tool-choice --tool-call-parser hermes` turn Qwen's `<tool_call>` text into OpenAI-style `tool_calls`, which the agents need.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

VLLM_PATTERN = "vllm serve|vllm.entrypoints"
BLACKBOX = f"{DRIVE_ROOT}/blackbox.log" if DRIVE_OK else "/content/blackbox.log"
# vLLM environment: use vLLM's own sampler instead of compiling FlashInfer's on the CPU at startup (the
# "Building JIT module sampling" step), and cap any other compile at 4 parallel jobs.
VLLM_ENV = {"VLLM_USE_FLASHINFER_SAMPLER": "0", "MAX_JOBS": "4"}


def ensure_blackbox():
    """Every 10 s append time, RAM, GPU memory and the last line of each server log to BLACKBOX.
    On Google Drive (USE_DRIVE = True) it survives a deleted runtime, so the last lines show what
    the machine was doing just before a disconnect."""
    if pids("blackbox.sh"):
        return
    os.makedirs(os.path.dirname(BLACKBOX), exist_ok=True)
    Path("/content/blackbox.sh").write_text(
        "while true; do {\n"
        "  echo \"$(date '+%F %T') | RAM $(free -m | awk '/Mem/{print $3\"/\"$2\" MB\"}')"
        " | GPU $(nvidia-smi --query-gpu=memory.used --format=csv,noheader | head -1)"
        " | load $(cut -d' ' -f1 /proc/loadavg)\"\n"
        "  for f in /content/vllm.log /content/gateway.log /content/tunnel.log; do\n"
        "    [ -f \"$f\" ] && echo \"   $(basename $f): $(tail -n 1 $f | cut -c1-200)\"\n"
        "  done\n"
        f"}} >> '{BLACKBOX}'; sleep 10; done\n")
    subprocess.Popen(["bash", "/content/blackbox.sh"], start_new_session=True,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("Black-box log:", BLACKBOX, "(on Google Drive)" if BLACKBOX.startswith("/content/drive") else
          "(local; set USE_DRIVE = True so it survives a deleted runtime)")


def vllm_failed(log):
    print("vLLM stopped. Last lines of the log:\n\n" + log)
    if "No module named" in log:
        hint = "a package is missing: Run all again (step 2 reinstalls)."
    elif "out of memory" in log.lower() or "KV cache" in log or "max_model_len" in log:
        hint = "not enough GPU memory: lower MAX_MODEL_LEN (e.g. 16384) in step 1, then Run all."
    else:
        hint = "see the log above."
    raise SystemExit("vLLM failed to start; " + hint)


def ensure_vllm(max_wait_minutes=15):
    if is_up(VLLM_HEALTH):
        print("vLLM is already running.")
        return
    if not os.path.exists(os.path.join(MODEL_DIR, ".complete")):
        raise SystemExit("The model is not on disk yet: run step 3a first.")
    running = pids(VLLM_PATTERN)
    if running and max(age_seconds(p) for p in running) < 20 * 60:
        print(f"vLLM is already starting (for {max(age_seconds(p) for p in running) // 60} min); waiting for it.")
    else:
        if running:
            print("Stopping a vLLM that has been starting for over 20 minutes ...")
        stop_processes(VLLM_PATTERN)                        # a stuck vLLM would keep the GPU memory
        require_free_port(VLLM_PORT, "VLLM_PORT")
        if not importable("vllm"):
            raise SystemExit("vLLM is not installed in this runtime: Run all again (step 2 reinstalls).")
        vllm_bin = shutil.which("vllm")
        cmd = [vllm_bin, "serve", MODEL_DIR] if vllm_bin else \
              [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_DIR]
        cmd += ["--host", "127.0.0.1", "--port", str(VLLM_PORT), "--served-model-name", MODEL_ID,
                "--dtype", DTYPE, "--max-model-len", str(MAX_MODEL_LEN),
                "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
                "--enable-auto-tool-choice", "--tool-call-parser", "hermes"]
        if FAST_START:
            cmd.append("--enforce-eager")
        # start_new_session: vLLM keeps running if the notebook kernel restarts
        subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT,
                         start_new_session=True, env=os.environ | VLLM_ENV)
        print("Starting vLLM in the background ... (log: /content/vllm.log)")
    started, last_line = time.time(), ""
    while not is_up(VLLM_HEALTH):
        time.sleep(5)
        elapsed = time.time() - started
        if not pids(VLLM_PATTERN):
            time.sleep(2)                                    # it may have just come up
            if not is_up(VLLM_HEALTH):
                vllm_failed(tail("/content/vllm.log"))
            break
        if int(elapsed) % 30 < 5:
            lines = [l.strip() for l in tail("/content/vllm.log", 3000).splitlines() if l.strip()]
            line = lines[-1][-160:] if lines else "(no log output yet)"
            if line != last_line:
                print(f"  {elapsed / 60:4.1f} min | {line}", flush=True)
                last_line = line
        if elapsed > max_wait_minutes * 60:
            raise SystemExit("vLLM is still not ready; it keeps loading in the background. "
                             "Run all again to keep waiting, or lower MAX_MODEL_LEN.")
    print(f"vLLM is ready ({MODEL_ID}).")


ensure_blackbox()
ensure_vllm()

## 4. Gateway: API key check, embeddings and chat proxy

A small FastAPI app on `127.0.0.1:8090` (not 8080: that port belongs to Colab's own runtime server, and taking it disconnects the notebook):
- every `/v1/*` request needs `Authorization: Bearer <API key>`
- `/v1/embeddings` runs the embedding model on the GPU (OpenAI format, float or base64)
- `/v1/chat/completions` is passed to vLLM unchanged, streaming included
- `/health` needs no key and reveals nothing

Request bodies are never logged. This cell only writes the file; step 5 starts it.

In [ ]:
%%writefile /content/gateway.py
"""OpenAI-compatible gateway for the SPARKS expense assistant (runs inside Colab)."""
import base64
import json
import os
import secrets

import httpx
import numpy as np
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import Response, StreamingResponse
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

API_KEY = os.environ["GATEWAY_API_KEY"]
VLLM_URL = os.environ.get("VLLM_URL", "http://127.0.0.1:8000")
EMBED_MODEL_ID = os.environ["EMBED_MODEL_ID"]
PORT = int(os.environ.get("GATEWAY_PORT", "8090"))
MAX_INPUTS = 256

embedder = SentenceTransformer(EMBED_MODEL_ID, device="cuda")
upstream = httpx.AsyncClient(base_url=VLLM_URL, timeout=httpx.Timeout(600, connect=10))
app = FastAPI(title="expense-assistant colab gateway", docs_url=None, redoc_url=None, openapi_url=None)


def require_key(request: Request):
    supplied = request.headers.get("authorization", "").encode()
    if not secrets.compare_digest(supplied, ("Bearer " + API_KEY).encode()):
        raise HTTPException(status_code=401, detail="invalid or missing API key")


class EmbeddingRequest(BaseModel):
    input: str | list[str]
    model: str | None = None
    encoding_format: str | None = None


@app.get("/health")
async def health():
    try:
        chat_ok = (await upstream.get("/health", timeout=5)).status_code == 200
    except httpx.HTTPError:
        chat_ok = False
    return {"status": "ok" if chat_ok else "degraded", "chat": chat_ok, "embeddings": True}


@app.get("/v1/models")
async def models(request: Request):
    require_key(request)
    body = (await upstream.get("/v1/models")).json()
    body.setdefault("data", []).append({"id": EMBED_MODEL_ID, "object": "model", "owned_by": "sentence-transformers"})
    return body


@app.post("/v1/embeddings")
def embeddings(payload: EmbeddingRequest, request: Request):
    require_key(request)
    inputs = [payload.input] if isinstance(payload.input, str) else payload.input
    if not inputs or len(inputs) > MAX_INPUTS:
        raise HTTPException(status_code=400, detail="input must hold 1 to %d strings" % MAX_INPUTS)
    vectors = embedder.encode(inputs, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    tokens = sum(len(ids) for ids in embedder.tokenizer(inputs)["input_ids"])
    data = []
    for index, vector in enumerate(vectors):
        if payload.encoding_format == "base64":
            value = base64.b64encode(np.asarray(vector, dtype="<f4").tobytes()).decode()
        else:
            value = vector.tolist()
        data.append({"object": "embedding", "index": index, "embedding": value})
    return {"object": "list", "data": data, "model": EMBED_MODEL_ID,
            "usage": {"prompt_tokens": tokens, "total_tokens": tokens}}


@app.post("/v1/chat/completions")
async def chat(request: Request):
    require_key(request)
    body = await request.body()
    try:
        stream = bool(json.loads(body or b"{}").get("stream"))
    except ValueError:
        raise HTTPException(status_code=400, detail="body must be JSON")
    headers = {"content-type": "application/json"}
    if not stream:
        reply = await upstream.post("/v1/chat/completions", content=body, headers=headers)
        return Response(reply.content, status_code=reply.status_code, media_type="application/json")
    reply = await upstream.send(upstream.build_request("POST", "/v1/chat/completions", content=body,
                                                       headers=headers), stream=True)

    async def relay():
        try:
            async for chunk in reply.aiter_raw():
                yield chunk
        finally:
            await reply.aclose()

    return StreamingResponse(relay(), status_code=reply.status_code,
                             media_type=reply.headers.get("content-type", "text/event-stream"))


if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=PORT, access_log=False)

## 5. Start the gateway and test it

Reuses a running gateway when it already accepts the current API key; otherwise (re)starts it.
Then tests structured output (as the app uses it), tool calling (as the agents will use it) and embeddings.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

def ensure_gateway():
    if is_up(f"{GATEWAY}/health") and is_up(f"{GATEWAY}/v1/models", headers=AUTH):
        print("Gateway already running with this API key.")
        return
    if not os.path.exists("/content/gateway.py"):
        raise SystemExit("/content/gateway.py is missing: run step 4 first.")
    if not is_up(VLLM_HEALTH):
        raise SystemExit("vLLM is not running: run steps 3a and 3b first.")
    stop_processes("/content/gateway.py")                  # our old gateway (maybe with an old key)
    require_free_port(GATEWAY_PORT, "GATEWAY_PORT")
    env = os.environ | {"GATEWAY_API_KEY": API_KEY, "EMBED_MODEL_ID": EMBED_MODEL_ID,
                        "VLLM_URL": f"http://127.0.0.1:{VLLM_PORT}", "GATEWAY_PORT": str(GATEWAY_PORT)}
    proc = subprocess.Popen([sys.executable, "/content/gateway.py"], env=env, start_new_session=True,
                            stdout=open("/content/gateway.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(100):                                    # first start downloads the embedding model
        if is_up(f"{GATEWAY}/health"):
            print("Gateway started.")
            return
        if proc.poll() is not None:
            print(tail("/content/gateway.log", 3000))
            raise SystemExit("Gateway failed to start; see the log above.")
        time.sleep(3)
    raise SystemExit("Gateway did not start in time; see /content/gateway.log")


def smoke_test(base):
    """Structured output, tool calling and embeddings through the gateway (base = local or public URL)."""
    assert requests.get(f"{base}/v1/models", timeout=10).status_code == 401, "requests without the key must be refused"
    schema = {"type": "object", "additionalProperties": False, "required": ["answer", "confidence"],
              "properties": {"answer": {"type": "string"}, "confidence": {"type": "string", "enum": ["low", "high"]}}}
    started = time.time()
    r = requests.post(f"{base}/v1/chat/completions", headers=AUTH, timeout=180, json={
        "model": MODEL_ID, "max_tokens": 200,
        "messages": [{"role": "system", "content": "Reply in JSON only."},
                     {"role": "user", "content": "In one sentence, what does a travel expense policy assistant do?"}],
        "response_format": {"type": "json_schema", "json_schema": {"name": "smoke", "schema": schema, "strict": True}}})
    if not r.ok:
        raise SystemExit(f"structured chat failed: HTTP {r.status_code} {r.text[:500]}")
    print(f"structured chat ({time.time() - started:.1f}s):", json.loads(r.json()["choices"][0]["message"]["content"]))
    r = requests.post(f"{base}/v1/chat/completions", headers=AUTH, timeout=180, json={
        "model": MODEL_ID, "max_tokens": 200, "tool_choice": "auto",
        "messages": [{"role": "user", "content": "What is the hotel cap for grade E in Bengaluru? Use the tool."}],
        "tools": [{"type": "function", "function": {
            "name": "get_cap", "description": "Look up a policy cap.",
            "parameters": {"type": "object", "required": ["grade", "city"],
                           "properties": {"grade": {"type": "string"}, "city": {"type": "string"}}}}}]})
    if not r.ok:
        raise SystemExit(f"tool call failed: HTTP {r.status_code} {r.text[:500]}")
    print("tool call:", r.json()["choices"][0]["message"].get("tool_calls"))
    r = requests.post(f"{base}/v1/embeddings", headers=AUTH, timeout=60,
                      json={"model": EMBED_MODEL_ID, "input": ["hotel cap Bengaluru", "local taxi limit"]})
    if not r.ok:
        raise SystemExit(f"embeddings failed: HTTP {r.status_code} {r.text[:500]}")
    print("embeddings:", len(r.json()["data"]), "vectors of dimension", len(r.json()["data"][0]["embedding"]))


ensure_gateway()
print("health:", requests.get(f"{GATEWAY}/health").json())
smoke_test(GATEWAY)

## 6. Public HTTPS tunnel

**Keeps the existing tunnel and URL** while it still works. A new tunnel is only started when the old one is gone.
- `cloudflared` (default): free quick tunnel, no account. It downloads Cloudflare's official binary into this Colab machine, not your laptop. The URL is random and changes on every fresh machine.
- `ngrok` with a free **static domain**: the URL **never changes**, so `.env` stays the same. Set `TUNNEL = "ngrok"` in step 1 and add `NGROK_AUTHTOKEN` and `NGROK_DOMAIN` as Colab Secrets.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

def ensure_tunnel():
    """Returns (public_url, changed). Reuses a working tunnel; starts a new one only when needed."""
    global PUBLIC_URL
    if not is_up(f"{GATEWAY}/health"):
        raise SystemExit("The gateway is not running: run step 5 first.")
    saved = Path(URL_FILE).read_text().strip() if os.path.exists(URL_FILE) else None
    if saved and is_up(f"{saved}/health"):
        PUBLIC_URL = saved
        print("Tunnel already running:", saved)
        return saved, False
    url = None
    if TUNNEL == "cloudflared":
        if not shutil.which("cloudflared"):
            subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
                            "cloudflared-linux-amd64"], check=True)
            os.chmod("/usr/local/bin/cloudflared", 0o755)
        stop_processes("cloudflared tunnel")
        proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", GATEWAY],
                                stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT,
                                start_new_session=True)
        for _ in range(60):
            found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tail("/content/tunnel.log", 20000))
            if found:
                url = found.group(0)
                break
            if proc.poll() is not None:
                break
            time.sleep(1)
    elif TUNNEL == "ngrok":
        if not importable("pyngrok"):
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
        from pyngrok import ngrok
        token, domain = colab_secret("NGROK_AUTHTOKEN"), colab_secret("NGROK_DOMAIN")
        if not token:
            raise SystemExit("TUNNEL = 'ngrok' needs an NGROK_AUTHTOKEN secret.")
        ngrok.kill()
        ngrok.set_auth_token(token)
        options = {"domain": domain} if domain else {}
        url = ngrok.connect(GATEWAY_PORT, "http", **options).public_url.replace("http://", "https://")
    if not url:
        print(tail("/content/tunnel.log", 2000))
        raise SystemExit("No tunnel URL; see the log above and Run all again.")
    for _ in range(45):                     # a new quick tunnel can take a little while to resolve
        if is_up(f"{url}/health"):
            break
        time.sleep(2)
    else:
        raise SystemExit(f"{url} does not answer yet; wait a minute and Run all again.")
    Path(URL_FILE).write_text(url)
    PUBLIC_URL = url
    changed = url != saved
    print("Tunnel started:", url, "(NEW URL: update .env)" if changed and saved else "")
    return url, changed


PUBLIC_URL, URL_CHANGED = ensure_tunnel()
print("public health:", requests.get(f"{PUBLIC_URL}/health", timeout=10).json())
print("authorised /v1/models:", requests.get(f"{PUBLIC_URL}/v1/models", headers=AUTH, timeout=10).status_code)

## 7. The `.env` block

Paste it into the project's `.env` (replacing an older "Hosted Qwen on Colab" block) **only when the URL or key changed**. Step 6 says so, and this cell repeats it.
Then on your laptop: `.venv\Scripts\python.exe -m scripts.check_hosted_model`, then restart the app (`run.bat`).
To go back to Claude Code, delete or comment out the block and restart the API.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

def print_env_block(reason=""):
    if reason:
        print("=" * 72 + f"\n{reason}\n" + "=" * 72)
    print(f"""# ---- Hosted Qwen on Colab (written {time.strftime('%Y-%m-%d %H:%M')} UTC)
EXPENSE_MODEL_PROVIDER=openai_compatible
EXPENSE_MODEL_BASE_URL={PUBLIC_URL}/v1
EXPENSE_MODEL_ID={MODEL_ID}
EXPENSE_MODEL_API_KEY_ENV=COLAB_API_KEY
EXPENSE_MODEL_MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS}
EXPENSE_MODEL_SUPPORTS_VISION=false
EXPENSE_EMBEDDINGS_PROVIDER=openai_compatible
EXPENSE_EMBEDDINGS_BASE_URL={PUBLIC_URL}/v1
EXPENSE_EMBEDDINGS_MODEL_ID={EMBED_MODEL_ID}
EXPENSE_EMBEDDINGS_API_KEY_ENV=COLAB_API_KEY
COLAB_API_KEY={API_KEY}
# ---- end hosted block""")


require("PUBLIC_URL", step=6)
print_env_block("URL CHANGED: paste this block into .env and restart the API" if globals().get("URL_CHANGED")
                else "Same URL as last time on this machine: no .env change needed if you already pasted it")

## 8. Monitor and auto-repair

Checks vLLM, the gateway and the tunnel every minute, and **restarts whatever is down** while the runtime is alive.
If the tunnel has to be replaced and the URL changes, it prints the new `.env` block.
It runs until you press ■. Pressing ■ only stops the monitor; the servers keep running.

In [ ]:
assert "require" in globals(), "Run step 1 (Settings and helpers) first: Colab forgets variables after a restart."

require("ensure_vllm", step="3b")
require("ensure_gateway", step=5)
require("ensure_tunnel", step=6)
require("print_env_block", step=7)
try:
    while True:
        vllm_ok, gateway_ok = is_up(VLLM_HEALTH), is_up(f"{GATEWAY}/health")
        tunnel_ok = is_up(f"{PUBLIC_URL}/health")
        print(time.strftime("%H:%M:%S"), f"vLLM={'up' if vllm_ok else 'DOWN'}",
              f"gateway={'up' if gateway_ok else 'DOWN'}", f"tunnel={'up' if tunnel_ok else 'DOWN'}", flush=True)
        try:
            if not vllm_ok:
                print("  repairing vLLM ...")
                ensure_vllm()
            if not is_up(f"{GATEWAY}/health"):
                print("  repairing gateway ...")
                ensure_gateway()
            if not is_up(f"{PUBLIC_URL}/health"):
                print("  repairing tunnel ...")
                PUBLIC_URL, changed = ensure_tunnel()
                if changed:
                    print_env_block("URL CHANGED: paste this block into .env and restart the API")
        except SystemExit as exc:            # keep monitoring; try again next minute
            print("  repair failed, retrying in a minute:", exc)
        except Exception as exc:
            print("  repair error, retrying in a minute:", type(exc).__name__, exc)
        time.sleep(60)
except KeyboardInterrupt:
    print("Monitor stopped; the servers keep running.")

## 9. Stop the servers (only when you mean it)

Does nothing unless you set `CONFIRM_STOP = True`. That way **Run all** never stops the servers by accident.
**Runtime → Disconnect and delete runtime** also stops everything and frees the GPU.

## Troubleshooting

| Problem | Fix |
|---|---|
| Disconnected, or anything failed | **Runtime → Run all.** Every step reuses what is still running. |
| Drive is getting full | Delete `MyDrive/colab_expense_assistant/pip-cache` (rebuilt on the next install) or `models/`. The API key and logs are tiny. |
| `FetchError … 8080-gpu-… 502` right after step 4 | Something killed or blocked Colab's own server on port 8080. Keep `GATEWAY_PORT = 8090`, and never run `fuser -k 8080` or start a server on 8080 in Colab. |
| Runtime keeps getting deleted (GPU 0 GB and model gone after reconnecting) | Open the black-box log (`MyDrive/colab_expense_assistant/blackbox.log`) and read the last lines before the gap. RAM near the limit points to a memory spike. If it always stops right after step 6, the tunnel is the likely trigger. |
| `NameError` or "run step 1 first" | You ran a single cell after a restart. Use Run all, or run step 1 first. |
| vLLM runs out of memory | Lower `MAX_MODEL_LEN` to 16384 or `GPU_MEMORY_UTILIZATION` to 0.75 in step 1, or use `Qwen/Qwen2.5-3B-Instruct`, then Run all. |
| vLLM still loading after 15 minutes | It keeps loading in the background; Run all again to keep waiting. |
| `bfloat16 is not supported` | You are on a T4; step 1 picks `half` automatically. Use an L4 if you can. |
| URL changes after every fresh machine | Use ngrok with a free static domain (`TUNNEL = "ngrok"`, `NGROK_AUTHTOKEN`, `NGROK_DOMAIN`) and a `COLAB_API_KEY` secret. |
| App shows **AI offline** or answers fail | The `.env` block is out of date (new URL or key). Paste the newest block and restart the API. |
| `/ready` shows `fallback_reason` | The app could not reach the embeddings endpoint at startup and is using its local copy. Fix the URL or key and restart the API. |

In [ ]:
CONFIRM_STOP = False      # set to True and run this cell to stop vLLM, the gateway and the tunnel
if not CONFIRM_STOP:
    print("Servers left running. Set CONFIRM_STOP = True in this cell to stop them.")
else:
    for pattern in ("cloudflared tunnel", "ngrok", "/content/gateway.py", "vllm serve|vllm.entrypoints"):
        subprocess.run(["pkill", "-f", pattern], capture_output=True)
    if os.path.exists("/content/.tunnel_url"):
        os.remove("/content/.tunnel_url")
    print("Stopped vLLM, gateway and tunnel.")